# Automated solar filament experiment

This kernel trains one reviewed configuration, tunes post-processing on fold 0, runs the organizer Self_Evaluation_Notebook PQ logic, and emits a controller-gated candidate.

In [ ]:
import json

EXPERIMENT = json.loads(r'''__EXPERIMENT_CONFIG__''')
EXPERIMENT

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/TaiTranDang145/Solar-Filament-Segmentation.git'
PROJECT_ROOT = Path('/kaggle/working/Solar-Filament-Segmentation')
if (PROJECT_ROOT / '.git').exists():
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm==1.0.29', 'ultralytics==8.4.115'], check=True)

annotation_file = next(Path('/kaggle/input').rglob('MAGFiLO_1.0_Annotations_kaggle2026_train.json'))
DATA_ROOT = annotation_file.parents[1]
OUTPUT_ROOT = Path('/kaggle/working/artifacts')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REVISION = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print('revision:', REVISION)
print('data:', DATA_ROOT)

In [ ]:
import torch
from solar_filament.data import audit_dataset

assert torch.cuda.is_available(), 'Kaggle GPU is required'
audit = audit_dataset(DATA_ROOT)
assert not audit.errors, audit.errors
print('GPU:', torch.cuda.get_device_name(0))
print(audit.as_dict())

In [ ]:
from dataclasses import asdict

IS_INSTANCE = EXPERIMENT.get('pipeline') == 'instance'
training_values = {'data_root': str(DATA_ROOT), **EXPERIMENT['training']}
if IS_INSTANCE:
    from solar_filament.instance_pipeline import InstanceConfig, run_instance_experiment
    training_values['output_dir'] = str(OUTPUT_ROOT / 'instance')
    train_config = InstanceConfig(**training_values)
    outcome = run_instance_experiment(
        train_config,
        confidence_thresholds=EXPERIMENT['confidence_thresholds'],
        mask_thresholds=EXPERIMENT['mask_thresholds'],
        min_areas=EXPERIMENT['min_areas'],
    )
else:
    from solar_filament.training import TrainConfig, train
    training_values['output_dir'] = str(OUTPUT_ROOT / 'training')
    train_config = TrainConfig(**training_values)
    checkpoint = train(train_config)
    print('checkpoint:', checkpoint)

In [ ]:
import math

if IS_INSTANCE:
    assert math.isclose(outcome.official_pq, outcome.self_evaluation_pq, abs_tol=1e-6)
    print('organizer self-evaluation PQ:', outcome.self_evaluation_pq)
    print(outcome)
else:
    from solar_filament.tuning import tune_checkpoint
    tuned_checkpoint = OUTPUT_ROOT / 'best-tuned.pt'
    tuning = tune_checkpoint(
        checkpoint,
        thresholds=EXPERIMENT['thresholds'],
        min_areas=EXPERIMENT['min_areas'],
        close_kernels=EXPERIMENT.get('close_kernels', [0]),
        output_path=tuned_checkpoint,
    )
    assert math.isclose(tuning.best.official_pq, tuning.self_evaluation_pq, abs_tol=1e-6)
    print('organizer self-evaluation PQ:', tuning.self_evaluation_pq)
    print(tuning.to_dict())

In [ ]:
submission = OUTPUT_ROOT / 'submission.csv'
if IS_INSTANCE:
    import shutil
    shutil.copy2(outcome.submission, submission)
else:
    from solar_filament.inference import infer_directory
    report = infer_directory(
        tuned_checkpoint,
        DATA_ROOT / 'test' / 'test_images',
        submission,
    )
    assert not report.errors
    print(report)

In [ ]:
candidate = {'revision': REVISION, 'experiment': EXPERIMENT['name'], 'training_config': asdict(train_config), 'submission': 'artifacts/submission.csv'}
if IS_INSTANCE:
    candidate.update(threshold=outcome.confidence_threshold, min_area=outcome.min_area, close_kernel=0, internal_pq=outcome.official_pq, self_evaluation_pq=outcome.self_evaluation_pq, checkpoint='artifacts/instance/best-yolo.pt')
else:
    candidate.update(threshold=tuning.best.threshold, min_area=tuning.best.min_area, close_kernel=tuning.best.close_kernel, internal_pq=tuning.best.official_pq, self_evaluation_pq=tuning.self_evaluation_pq, checkpoint='artifacts/best-tuned.pt')
(OUTPUT_ROOT / 'candidate.json').write_text(json.dumps(candidate, indent=2) + '\n')
candidate